In [1]:
import numpy as np 
import pandas as pd 
import seaborn as sns 
from matplotlib import pyplot as plt 
import random
import warnings
warnings.filterwarnings('ignore') 

In [2]:
# node class 
class Node: 
    def __init__(self , feature_index = None , threshold = None , left = None , right = None , * , value = None):
        self.feature_index = feature_index
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value
        
    def is_leaf(self):
        return self.value is not None

In [3]:
from collections import Counter
class DecisionTreeClassifier: 
    def __init__(self , max_depth = 10 , min_samples_split = 2 , min_samples_leaf = 1 , criterion = 'gini'): 
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.criterion = criterion 
        self.root = None 

    def fit(self , X , y , feature_types): 
        # find total number of classes 
        self.n_classes = len(set(y))
        # store the feature types
        self.feature_types = feature_types

        # Build the tree 
        self.root = self._build_tree(X , y , curr_depth = 0)

    def _build_tree(self , X , y , curr_depth = 0): 
        n_samples , n_features = X.shape
        n_labels = len(set(y)) 
        # stopping conditions 
        if n_samples <= self.min_samples_split or curr_depth >= self.max_depth or n_labels == 1: 
            # stop , make this one a leaf node 
            leaf_value = self._most_common_label(y)
            return Node(value = leaf_value)
            
        # find best feature along with threshold to do split 
        best_feature , best_threshold = self._best_split(X , y)
        # if no best split available 
        if best_feature is None: 
            # stop , make this one a leaf node 
            leaf_value = self._most_common_label(y)
            return Node(value = leaf_value) 
            
        # do the split
        left_idx , right_idx = self._split(X[ : , best_feature] , best_threshold , self.feature_types[best_feature])
        # if pure split 
        if len(left_idx) < self.min_samples_leaf or len(right_idx) < self.min_samples_leaf: 
            # stop , make this one a leaf node 
            leaf_value = self._most_common_label(y)
            return Node(value = leaf_value)
            
        # get the samples of left child and right child 
        # recursively build the left and right tree 
        
        left = self._build_tree(X[left_idx] , y[left_idx] , curr_depth + 1)
        right = self._build_tree(X[right_idx] , y[right_idx] , curr_depth + 1)

        return Node(best_feature , best_threshold , left , right)

    def _best_split(self , X , y): 
        best_gain = -1 
        best_feature , best_threshold = None , None 

        total_features = X.shape[1]
        for feature_index in range(total_features): 
            values = X[ : , feature_index]
            feature_type = self.feature_types[feature_index]

            if feature_type == 'numerical': 
                # find all the threshold values
                thresholds = np.unique(values.astype(float))
                for threshold in thresholds: 
                    gain = self._information_gain(y , values , threshold , feature_type)

                    if gain > best_gain: 
                       best_gain = gain
                       best_feature = feature_index
                       best_threshold = threshold
            elif feature_type == 'categorical':
                categories = np.unique(values)
                for category in categories: 
                    gain = self._information_gain(y , values , category , feature_type)

                    if gain > best_gain: 
                        best_gain = gain
                        best_feature = feature_index
                        best_threshold = category
        
        return best_feature , best_threshold
    
    def _split(self , column , threshold , feature_type): 
        if feature_type == 'numerical': 
            left_idx = np.where(column.astype(float) <= float(threshold))[0]
            right_idx = np.where(column.astype(float) > float(threshold))[0]
        else:
            left_idx = np.where(column == threshold)[0]
            right_idx = np.where(column != threshold)[0]

        return left_idx , right_idx
    def _information_gain(self , y , feature_column , threshold , feature_type):
         # parents impurity
         parent_impurity = self._impurity(y)
         # find left and right child
         left_idx , right_idx = self._split(feature_column , threshold , feature_type)
         if len(left_idx) == 0 or len(right_idx) == 0: 
             return 0
         # normal impurity of both child
         left_impurity = self._impurity(y[left_idx])
         right_impurity = self._impurity(y[right_idx])
         n , n_left , n_right = len(y) , len(left_idx) , len(right_idx)
         # find the weighted impurity
         child_entropy = (n_left / n) * left_impurity + (n_right / n) * right_impurity
         # find the information gain 
         return parent_impurity - child_entropy
        
    def _impurity(self , y):
        prob = np.bincount(y) / len(y) 
        
        if self.criterion == 'gini': 
            return 1 - np.sum(prob ** 2)    
        elif self.criterion == 'entropy':
            return -np.sum([p * np.log2(p) for p in prob if p > 0]) 
        else: 
            raise RuntimeError("Unkown criterion")
            
    def _most_common_label(self , y): 
        counter = Counter(y)
        return counter.most_common(1)[0][0]
    def predict(self , X): 
        return np.array([self._row_predict(self.root , x) for x in X])
    def _row_predict(self , node , x): 
        if node.is_leaf(): 
            return node.value

        val = x[node.feature_index]

        if self.feature_types[node.feature_index] == 'numerical': 
           if float(val) <= float(node.threshold): 
               return self._row_predict(node.left , x)
           else:
               return self._row_predict(node.right , x)
        else:
            if val == node.threshold: 
                return _row_predict(node.left , x)
            else:
                return _row_predict(node.right , x)

In [5]:
class MyRandomForestClassifier:
    def __init__(self , n_estimators = 100 , max_depth = 10 , min_samples_split = 2 , 
                 min_samples_leaf = 1 , criterion = 'gini' , max_features = 'sqrt' , samples_size = 1.0): 
        self.n_estimators = n_estimators # number of decision tree used
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.criterion = criterion
        self.max_features = max_features # can be int , 'sqrt' , 'log2' , or None(all)
        self.samples_size = samples_size
        self.trees = [] 
        self.feature_types = None

    def fit(self , X_train , y_train , feature_types):
        
        self.feature_types = feature_types
        self.n_features = X_train.shape[1]

        for _ in range(self.n_estimators): 
            # Bootstrap the samples
            X_sample , y_sample , selected_feature_types , feature_indices = self._bootstrap_sample(X_train , y_train , feature_types)
           
            # Train a Tree
            tree = DecisionTreeClassifier(
                max_depth = self.max_depth, 
                min_samples_split = self.min_samples_split, 
                min_samples_leaf = self.min_samples_leaf, 
                criterion = self.criterion
            )
       
            # Fit the model 
            tree.fit(X_sample , y_sample , feature_types = selected_feature_types)
            # store the model 
            self.trees.append((tree , feature_indices)) 

    def _bootstrap_sample(self , X , y , feature_types): 
        n_rows , n_columns = X.shape
        sample_n = int(self.samples_size * n_rows)

        # row sampling 
        row_indices = np.random.choice(n_rows , size = sample_n , replace = True) 
        X_sample = X[row_indices]
        y_sample = y[row_indices]

        # Feature sampling 
        if isinstance(self.max_features , int): 
            feature_indices = np.random.choice(n_columns , size = self.max_features , replace = False) 
        elif self.max_features == 'sqrt': 
             feature_indices = np.random.choice(n_columns , size = int(np.sqrt(n_columns)) , replace = False)
        elif self.max_features == 'log2': 
            feature_indices = np.random.choice(n_columns , size = int(np.log2(n_columns)) , replace = False) 
        else: # use all features 
            feature_indices = np.arange(n_columns)

        return X_sample[ : , feature_indices] , y_sample , [feature_types[i] for i in feature_indices] , feature_indices
    def predict(self, X):
        tree_preds = []

        for tree, feature_indices in self.trees:
            X_subset = X[ : , feature_indices]
            preds = tree.predict(X_subset)
            tree_preds.append(preds)
    
        # Transpose so we can do majority voting per sample
        tree_preds = np.array(tree_preds)  # shape: (n_trees, n_samples)
        return np.apply_along_axis(self._majority_vote, axis = 0, arr = tree_preds)

    def _majority_vote(self, predictions):
        return Counter(predictions).most_common(1)[0][0]
        

In [6]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer()
X = data.data
y = data.target
feature_names = data.feature_names

In [7]:
X

array([[1.799e+01, 1.038e+01, 1.228e+02, ..., 2.654e-01, 4.601e-01,
        1.189e-01],
       [2.057e+01, 1.777e+01, 1.329e+02, ..., 1.860e-01, 2.750e-01,
        8.902e-02],
       [1.969e+01, 2.125e+01, 1.300e+02, ..., 2.430e-01, 3.613e-01,
        8.758e-02],
       ...,
       [1.660e+01, 2.808e+01, 1.083e+02, ..., 1.418e-01, 2.218e-01,
        7.820e-02],
       [2.060e+01, 2.933e+01, 1.401e+02, ..., 2.650e-01, 4.087e-01,
        1.240e-01],
       [7.760e+00, 2.454e+01, 4.792e+01, ..., 0.000e+00, 2.871e-01,
        7.039e-02]], shape=(569, 30))

In [8]:
# hee all the features are numerical 
feature_types = ['numerical'] * X.shape[1] 

In [9]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X , y, test_size = 0.2, random_state = 42)

In [10]:
clf = MyRandomForestClassifier(
    n_estimators = 10 ,
    max_depth = 10 ,
    min_samples_split = 2 ,
    min_samples_leaf = 1 ,
    criterion = 'gini' ,
    max_features = 'sqrt' ,
    samples_size = 0.8
)

In [11]:
X_train.shape

(455, 30)

In [13]:
feature_types = ['numerical'] * X_train.shape[1] 
clf.fit(X_train , y_train , feature_types)

In [14]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test , clf.predict(X_test))

0.9473684210526315